# Legendary 8h Benchmark Survey — Colab A100

Full 11-model x 9-benchmark clone-detection matrix in a single ~8h Colab session on an **A100 40GB**.

**Tactics**
- fp16 throughout
- Per-benchmark `sample_pct` calibrated to ~6–8k train pairs (small benches use 100%)
- Shorter `max_length=384`, larger batches (`train=16`, `eval=32`), `epochs=2`
- `bootstrap_resamples=500`
- Hard wall-clock budget per cell; over-budget cells are skipped (not failed)
- Resumable: skips cells already on disk
- Auto-saves results to Drive

Order: GPU check -> mount Drive -> clone & install -> datasets -> matrix -> summary.

## 1. GPU check (must be A100)

In [ ]:
!nvidia-smi | head -20
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100.'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
if 'A100' not in name:
    print('WARNING: not an A100 — timing budget assumes A100 40GB.')

## 2. Mount Drive (results survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/small_code_models_legendary'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 3. Clone repo & install
Edit `REPO_URL` if you forked. The repo is installed in editable mode.

In [ ]:
REPO_URL = 'https://github.com/jorge-martinez-gil/small-code-models.git'  # <-- edit if needed
BRANCH = 'main'
WORK = '/content/small-code-models'
import os, subprocess
if not os.path.isdir(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, WORK], check=True)
%cd $WORK
!pip -q install --upgrade pip
!pip -q install -e .

## 4. Symlink datasets/results to Drive (resumable)

In [ ]:
import os
for sub in ('datasets', 'results', '.hf_cache'):
    drive_dir = os.path.join(DRIVE_ROOT, sub)
    local = os.path.join(WORK, sub)
    os.makedirs(drive_dir, exist_ok=True)
    if os.path.islink(local) or not os.path.exists(local):
        if os.path.exists(local):
            os.remove(local)
        os.symlink(drive_dir, local)
    print(local, '->', os.readlink(local) if os.path.islink(local) else local)

## 5. Download & normalize datasets

In [ ]:
!python scripts/download_datasets.py --dataset all --output_root datasets --hf_cache_dir .hf_cache --skip_existing
!python scripts/normalize_local_datasets.py --input_root datasets --output_root datasets --dataset all

## 6. Legendary config

Per-benchmark `sample_pct` targets ~6–8k train pairs. Adjust `WALL_BUDGET_S` to fit your session.

In [ ]:
import time

MODELS = [
    'codebert', 'graphcodebert', 'unixcoder',
    'codet5_small', 'codeberta_small',
    'codegpt_py', 'codegpt_java',
    'codet5', 'codet5p_220m',
    'cotext_1_cc', 'cotext_2_cc',
]

# Per-benchmark sample_pct calibrated against train.txt line counts
BENCH_PCT = {
    'bcb':                0.9,    # 901k -> ~8k
    'poj104':             6.0,    # 130k -> ~8k
    'gcj':                1.1,    # 726k -> ~8k
    'karnalim':           100.0,  # 322   -> all
    'poolc':              0.15,   # 5.36M -> ~8k
    'codenet':            0.5,    # 1.58M -> ~8k
    'semanticclonebench': 100.0,
    'gptclonebench':      100.0,
    'clcdsa':             100.0,
}

# Larger models (t5-base scale) get a slimmer slice to stay on schedule
HEAVY = {'codet5', 'codet5p_220m', 'cotext_1_cc', 'cotext_2_cc'}
HEAVY_FACTOR = 0.6

HYPER = dict(
    epochs=2,
    seed=42,
    max_length=384,
    train_batch_size=16,
    eval_batch_size=32,
    bootstrap_resamples=500,
)

# Per-cell wall-clock budget. Cells that would push past WALL_BUDGET_S are skipped.
PER_CELL_BUDGET_S = 9 * 60       # soft target per cell
WALL_BUDGET_S    = 7.5 * 3600    # total budget; leaves 30 min for summary + Drive sync

print('Total cells:', len(MODELS) * len(BENCH_PCT))
print('Soft per-cell budget (s):', PER_CELL_BUDGET_S)
print('Wall budget (h):', WALL_BUDGET_S / 3600)

## 7. Run the matrix (resumable, time-bounded)

In [ ]:
import os, json, subprocess, time, shutil
from pathlib import Path

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
DATASETS = Path('datasets')
status_path = RESULTS / 'run_status.tsv'
if not status_path.exists():
    status_path.write_text('model\tbenchmark\tstatus\tseconds\toutput_dir\n')

def cell_done(out_dir: Path) -> bool:
    # Treat a cell as done if test_results.json exists (written at end of run)
    return (out_dir / 'test_results.json').exists() or (out_dir / 'metrics.json').exists()

def append_status(model, bench, status, seconds, out_dir):
    with status_path.open('a') as f:
        f.write(f'{model}\t{bench}\t{status}\t{seconds:.1f}\t{out_dir}\n')

start = time.time()
skipped_for_time = []

available = [b for b in BENCH_PCT if (DATASETS / b / 'train.txt').exists()]
print('Available benchmarks:', available)

for model in MODELS:
    for bench in available:
        out_dir = RESULTS / f'{model}_{bench}'
        out_dir.mkdir(parents=True, exist_ok=True)
        if cell_done(out_dir):
            print(f'[skip-done] {model}/{bench}')
            continue

        elapsed = time.time() - start
        if elapsed > WALL_BUDGET_S:
            print(f'[skip-time] {model}/{bench}  (wall budget exhausted)')
            skipped_for_time.append((model, bench))
            append_status(model, bench, 'SKIP_TIME', 0.0, str(out_dir))
            continue

        pct = BENCH_PCT[bench]
        if model in HEAVY and pct < 100.0:
            pct = max(0.05, pct * HEAVY_FACTOR)

        cmd = [
            'python', 'scripts/run_clone_experiment.py',
            '--model', model,
            '--benchmark', bench,
            '--data_dir', f'datasets/{bench}',
            '--output_dir', str(out_dir),
            '--sample_pct', f'{pct}',
            '--epochs', str(HYPER['epochs']),
            '--seed', str(HYPER['seed']),
            '--max_length', str(HYPER['max_length']),
            '--train_batch_size', str(HYPER['train_batch_size']),
            '--eval_batch_size', str(HYPER['eval_batch_size']),
            '--bootstrap_resamples', str(HYPER['bootstrap_resamples']),
            '--fp16',
        ]
        print(f'\n=== [{(time.time()-start)/60:5.1f}m] {model} / {bench}  (sample_pct={pct}) ===')
        t0 = time.time()
        try:
            subprocess.run(cmd, check=True, timeout=PER_CELL_BUDGET_S * 2)
            dt = time.time() - t0
            status = 'OK' if cell_done(out_dir) else 'NO_OUTPUT'
        except subprocess.TimeoutExpired:
            dt = time.time() - t0
            status = 'TIMEOUT'
            print(f'  -> TIMEOUT after {dt:.0f}s')
        except subprocess.CalledProcessError as e:
            dt = time.time() - t0
            status = f'FAIL({e.returncode})'
            print(f'  -> FAIL after {dt:.0f}s')
        append_status(model, bench, status, dt, str(out_dir))
        print(f'  -> {status} in {dt:.0f}s')

print('\nTotal wall:', f'{(time.time()-start)/60:.1f} min')
print('Skipped for time:', skipped_for_time)

## 8. Summaries & pairwise comparisons

In [ ]:
!python scripts/summarize_results.py results || echo 'summarize_results.py failed (continuing)'

import os
BASELINE = 'codebert'
CANDIDATES = ['graphcodebert', 'unixcoder', 'codet5_small']
os.makedirs('results/comparisons', exist_ok=True)
for bench in BENCH_PCT:
    base = f'results/{BASELINE}_{bench}/predictions.jsonl'
    if not os.path.exists(base):
        continue
    for cand in CANDIDATES:
        cand_path = f'results/{cand}_{bench}/predictions.jsonl'
        if not os.path.exists(cand_path):
            continue
        out = f'results/comparisons/{cand}_vs_{BASELINE}_{bench}.json'
        !python scripts/compare_predictions.py {base} {cand_path} --metric f1 --bootstrap_resamples 500 --seed 42 --output {out} || true

## 9. Final scoreboard

In [ ]:
import json, glob, pandas as pd
rows = []
for path in sorted(glob.glob('results/*/test_results.json')) + sorted(glob.glob('results/*/metrics.json')):
    try:
        with open(path) as f:
            d = json.load(f)
    except Exception:
        continue
    name = path.split('/')[-2]
    if '_' in name:
        model, _, bench = name.partition('_')
    else:
        model, bench = name, ''
    flat = {'model': model, 'benchmark': bench}
    for k, v in d.items():
        if isinstance(v, (int, float)):
            flat[k.replace('eval_', '')] = v
    rows.append(flat)

df = pd.DataFrame(rows)
if not df.empty and 'f1' in df.columns:
    pivot = df.pivot_table(index='model', columns='benchmark', values='f1', aggfunc='first')
    print('\n=== F1 scoreboard ===')
    print(pivot.round(3).to_string())
    pivot.to_csv('results/scoreboard_f1.csv')
df.to_csv('results/scoreboard_raw.csv', index=False)
print('\nWrote results/scoreboard_f1.csv and results/scoreboard_raw.csv')

## 10. Sync to Drive (insurance)

In [ ]:
# results/ is already a symlink into Drive; this just forces a flush
import subprocess
subprocess.run(['sync'])
print('Done. Find everything under:', DRIVE_ROOT)